# 1. Le JSON, sans réseau

In [63]:
import os
print(os.getcwd())

c:\Users\Julie\code\json-api-geocodage\notebooks


In [64]:
import os

print("je m'execute depuis :", os.getcwd())
print("le dossier data existe :", os.path.isdir("../data"))
print("le fichier existe     :", os.path.isfile("../data/restaurants.json"))
print("contenu de ../data    :", os.listdir("../data"))

je m'execute depuis : c:\Users\Julie\code\json-api-geocodage\notebooks
le dossier data existe : True
le fichier existe     : True
contenu de ../data    : ['restaurants.json', 'restaurants_geocodes.csv', 'villes.json']


In [65]:
import json

with open("../data/restaurants.json", encoding="utf-8") as f:
    data = json.load(f)

print(type(data))
print(data["ville"], "-", data["pays"])
print("nombre de restaurants :", len(data["restaurants"]))

<class 'dict'>
Catane - Italie
nombre de restaurants : 4


In [66]:
print(data["restaurants"][0]["nom"])
print(data["restaurants"][0]["notes"][0])

Gelateria Zio Pietro dal 1964
4.6


In [67]:
for r in data["restaurants"]:
    moyenne = sum(r["notes"]) / len(r["notes"])
    print(f"{r['nom']:32} {moyenne:.2f}")

Gelateria Zio Pietro dal 1964    4.70
A Casa d'Amici                   4.45
La Bitta                         4.15
Steak House                      4.00


In [68]:
print(json.dumps(data["restaurants"][1], ensure_ascii=False, indent=2))

{
  "nom": "A Casa d'Amici",
  "adresse": "Via Fischetti, 14, 95131 Catania CT",
  "type": "trattoria",
  "notes": [
    4.4,
    4.5
  ]
}


# 2. Du JSON au DataFrame

## Cours

In [69]:
import pandas as pd

df = pd.json_normalize(data)
print(df.shape)
print(df.columns.tolist())

(1, 4)
['ville', 'pays', 'releve', 'restaurants']


In [70]:
df = pd.json_normalize(data, record_path="restaurants")
df

,nom,adresse,type,notes
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",glacier,"[4.6, 4.8, 4.7]"
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",trattoria,"[4.4, 4.5]"
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",poisson,"[4.2, 4.0, 4.3, 4.1]"
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",grill,"[3.9, 4.1]"


In [71]:
df = pd.json_normalize(data, record_path="restaurants", meta=["ville", "pays"])
df

,nom,adresse,type,notes,ville,pays
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",glacier,"[4.6, 4.8, 4.7]",Catane,Italie
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",trattoria,"[4.4, 4.5]",Catane,Italie
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",poisson,"[4.2, 4.0, 4.3, 4.1]",Catane,Italie
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",grill,"[3.9, 4.1]",Catane,Italie


# Challenge 1 — Un JSON à deux étages 

In [72]:
with open("../data/villes.json", encoding="utf-8") as f:
    villes_data = json.load(f)

In [73]:

villes_data

{'pays': 'Italie',
 'releve': '2026-11-05',
 'villes': [{'nom': 'Catane',
   'region': 'Sicile',
   'habitants': 298762,
   'restaurants': [{'nom': 'Gelateria Zio Pietro dal 1964',
     'type': 'glacier',
     'notes': [4.6, 4.8, 4.7]},
    {'nom': "A Casa d'Amici", 'type': 'trattoria', 'notes': [4.4, 4.5]},
    {'nom': 'La Bitta', 'type': 'poisson', 'notes': [4.2, 4.0, 4.3, 4.1]}]},
  {'nom': 'Palerme',
   'region': 'Sicile',
   'habitants': 630828,
   'restaurants': [{'nom': 'Antica Focacceria',
     'type': 'trattoria',
     'notes': [4.3, 4.1, 4.4]},
    {'nom': 'Ferro di Cavallo', 'type': 'trattoria', 'notes': [4.5, 4.6]}]},
  {'nom': 'Syracuse',
   'region': 'Sicile',
   'habitants': 121605,
   'restaurants': [{'nom': 'Don Camillo',
     'type': 'poisson',
     'notes': [4.7, 4.8, 4.6, 4.9]},
    {'nom': 'Gelateria Bianca', 'type': 'glacier', 'notes': [4.2]}]}]}

In [74]:
print("Nombre de villes :", len(villes_data["villes"]))
print("Nom de la deuxieme ville :", villes_data["villes"][1]["nom"])
print("Nombre de restaurants à Syracuse", len(villes_data["villes"][2]["restaurants"]))
print("Nom du premier restaurant de Palerme :", villes_data["villes"][1]["restaurants"][0]["nom"])

Nombre de villes : 3
Nom de la deuxieme ville : Palerme
Nombre de restaurants à Syracuse 2
Nom du premier restaurant de Palerme : Antica Focacceria


In [75]:
for ville in villes_data["villes"]:
    print(ville["nom"], ":" , len(ville["restaurants"]), "restaurants")

Catane : 3 restaurants
Palerme : 2 restaurants
Syracuse : 2 restaurants


In [76]:
df_villes = pd.json_normalize(
    villes_data,
    record_path=["villes", "restaurants"],
    meta=[
        ["villes", "nom"],
        ["villes", "region"],
        ["villes", "habitants"]
    ]
)

df_villes

,nom,type,notes,villes.nom,villes.region,villes.habitants
0,Gelateria Zio Pietro dal 1964,glacier,"[4.6, 4.8, 4.7]",Catane,Sicile,298762
1,A Casa d'Amici,trattoria,"[4.4, 4.5]",Catane,Sicile,298762
2,La Bitta,poisson,"[4.2, 4.0, 4.3, 4.1]",Catane,Sicile,298762
3,Antica Focacceria,trattoria,"[4.3, 4.1, 4.4]",Palerme,Sicile,630828
4,Ferro di Cavallo,trattoria,"[4.5, 4.6]",Palerme,Sicile,630828
5,Don Camillo,poisson,"[4.7, 4.8, 4.6, 4.9]",Syracuse,Sicile,121605
6,Gelateria Bianca,glacier,[4.2],Syracuse,Sicile,121605


In [77]:
moyennes = []

for notes in df_villes["notes"]:
    moyenne = sum(notes) / len(notes)
    moyennes.append(round(moyenne, 2))

df_villes["note_moyenne"] = moyennes

df_villes

,nom,type,notes,villes.nom,villes.region,villes.habitants,note_moyenne
0,Gelateria Zio Pietro dal 1964,glacier,"[4.6, 4.8, 4.7]",Catane,Sicile,298762,4.70
1,A Casa d'Amici,trattoria,"[4.4, 4.5]",Catane,Sicile,298762,4.45
2,La Bitta,poisson,"[4.2, 4.0, 4.3, 4.1]",Catane,Sicile,298762,4.15
3,Antica Focacceria,trattoria,"[4.3, 4.1, 4.4]",Palerme,Sicile,630828,4.27
4,Ferro di Cavallo,trattoria,"[4.5, 4.6]",Palerme,Sicile,630828,4.55
5,Don Camillo,poisson,"[4.7, 4.8, 4.6, 4.9]",Syracuse,Sicile,121605,4.75
6,Gelateria Bianca,glacier,[4.2],Syracuse,Sicile,121605,4.20


In [78]:
df_villes.sort_values(by="note_moyenne", ascending=False).head(1)

,nom,type,notes,villes.nom,villes.region,villes.habitants,note_moyenne
5,Don Camillo,poisson,"[4.7, 4.8, 4.6, 4.9]",Syracuse,Sicile,121605,4.75


In [79]:
df_villes.groupby("type").agg("nom").count().sort_values(ascending=False)

type
trattoria    3
glacier      2
poisson      2
Name: nom, dtype: int64

# 3. Une API, c'est un JSON au bout d'une URL

## Cours

In [80]:
import requests

url = "https://api-adresse.data.gouv.fr/search/"
r = requests.get(url, params={"q": "20 avenue de Ségur 75007 Paris", "limit": 1})

print(r.status_code)

200


In [81]:
reponse = r.json()
print(type(reponse))
print(reponse.keys())

<class 'dict'>
dict_keys(['type', 'features', 'query'])


In [82]:
r = requests.get(url, params={"q": "20 avenue de Ségur 75007 Paris", "limit": 1})
print(r.url)

https://api-adresse.data.gouv.fr/search/?q=20+avenue+de+S%C3%A9gur+75007+Paris&limit=1


In [83]:
headers = {"User-Agent": "bootcamp-simplon-da/1.0 (prenom.nom@exemple.fr)"}
r = requests.get(url, params={"q": "Catane"}, headers=headers)

print(r.status_code)
print(r.json()["features"][0]["properties"]["label"])

ConnectionError: ('Connection aborted.', ConnectionResetError(10054, 'Une connexion existante a dû être fermée par l’hôte distant', None, 10054, None))

In [ ]:
import requests

def interroger(url, params, headers=None):
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        r.raise_for_status()
        return r.json()
    except requests.exceptions.Timeout:
        print("Trop lent, on abandonne cette requête")
    except requests.exceptions.HTTPError as e:
        print("Le serveur a répondu une erreur :", e)
    except requests.exceptions.RequestException as e:
        print("Problème réseau :", e)
    return None

In [ ]:
print(interroger("https://api-adresse.data.gouv.fr/search/", {"q": "Lille", "limit": 1}) is not None)
print(interroger("https://api-adresse.data.gouv.fr/nimportequoi", {"q": "Lille"}))


True
Le serveur a répondu une erreur : 404 Client Error: Not Found for url: https://api-adresse.data.gouv.fr/nimportequoi?q=Lille
None


# Challenge 2 — Interroger une API en série

In [ ]:
adresses = [
    "20 avenue de Ségur 75007 Paris",
    "Grand Place, Lille",
    "1 rue de la Paix, Paris",
    "place Bellecour, Lyon",
    "zzzzz qwerty 99999",
]

In [ ]:
r = requests.get(
    url,
    params={"q": adresses[0], "limit": 1},
    timeout=10
)

print("Code HTTP :", r.status_code)

reponse = r.json()

print("Label :", reponse["features"][0]["properties"]["label"])
print("Score :", reponse["features"][0]["properties"]["score"])

Code HTTP : 200
Label : 20 Avenue de Ségur 75007 Paris
Score : 0.9716281818181818


In [ ]:
resultats = []

for adresse in adresses:
    reponse = interroger(url, {"q": adresse, "limit": 1})

    if reponse is not None and len(reponse["features"]) > 0:
        p = reponse["features"][0]["properties"]
    else:
        p = {"label": None, "score": None, "postcode": None, "city": None}

    resultats.append({
        "adresse": adresse,
        "label": p["label"],
        "score": p["score"],
        "postcode": p["postcode"],
        "city": p["city"]
    })

print(resultats)

[{'adresse': '20 avenue de Ségur 75007 Paris', 'label': '20 Avenue de Ségur 75007 Paris', 'score': 0.9716281818181818, 'postcode': '75007', 'city': 'Paris'}, {'adresse': 'Grand Place, Lille', 'label': 'Place de Gand 59800 Lille', 'score': 0.5682435064935065, 'postcode': '59800', 'city': 'Lille'}, {'adresse': '1 rue de la Paix, Paris', 'label': '1 Rue de la Paix 75002 Paris', 'score': 0.9640599999999998, 'postcode': '75002', 'city': 'Paris'}, {'adresse': 'place Bellecour, Lyon', 'label': 'Place Bellecour 69002 Lyon', 'score': 0.9673672727272726, 'postcode': '69002', 'city': 'Lyon'}, {'adresse': 'zzzzz qwerty 99999', 'label': None, 'score': None, 'postcode': None, 'city': None}]


In [ ]:
df_adresses = pd.DataFrame(resultats)       
df_adresses

,adresse,label,score,postcode,city
0,20 avenue de Ségur 75007 Paris,20 Avenue de Ségur 75007 Paris,0.971628,75007,Paris
1,"Grand Place, Lille",Place de Gand 59800 Lille,0.568244,59800,Lille
2,"1 rue de la Paix, Paris",1 Rue de la Paix 75002 Paris,0.964060,75002,Paris
3,"place Bellecour, Lyon",Place Bellecour 69002 Lyon,0.967367,69002,Lyon
4,zzzzz qwerty 99999,NaN,NaN,NaN,NaN



1. Combien d'adresses ont été trouvées ?

    4 adresses valide sur 5 ont été trouvées.

2. Quelle adresse a le score le plus bas ? Pourquoi ?

    Grand Place, Lille, avec un score de 0,568244.
    il est inscrit "Place de Gand 59800 Lille", alors que nous recherchions "Grand Place, Lille". 

3. Le label renvoyé correspond-il toujours à l'adresse demandée ?

    L'exemple de Lille montre que non

# 4. Géocoder : d'une adresse à un point

In [ ]:
import requests

url = "https://api-adresse.data.gouv.fr/search/"
r = requests.get(url, params={"q": "20 avenue de Ségur 75007 Paris", "limit": 1})
reponse = r.json()

import json
print(json.dumps(reponse, ensure_ascii=False, indent=2)[:600])

{
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "geometry": {
        "type": "Point",
        "coordinates": [
          2.308628,
          48.850699
        ]
      },
      "properties": {
        "label": "20 Avenue de Ségur 75007 Paris",
        "score": 0.9716281818181818,
        "housenumber": "20",
        "id": "75107_8909_00020",
        "banId": "c575af14-f697-464c-b70b-914cedb239a4",
        "name": "20 Avenue de Ségur",
        "postcode": "75007",
        "citycode": "75107",
        "x": 649266.35,
        "y": 6861406.23,
        "city": 


In [ ]:
f = reponse["features"][0]
lon, lat = f["geometry"]["coordinates"]
print("longitude :", lon, " latitude :", lat)
print("label     :", f["properties"]["label"])

longitude : 2.308628  latitude : 48.850699
label     : 20 Avenue de Ségur 75007 Paris


In [ ]:
import pandas as pd

df = pd.json_normalize(reponse, record_path="features")
print(df.columns.tolist())

['type', 'geometry.type', 'geometry.coordinates', 'properties.label', 'properties.score', 'properties.housenumber', 'properties.id', 'properties.banId', 'properties.name', 'properties.postcode', 'properties.citycode', 'properties.x', 'properties.y', 'properties.city', 'properties.district', 'properties.context', 'properties.type', 'properties.importance', 'properties.depcode', 'properties.street', 'properties._type']


In [ ]:
df["lon"] = df["geometry.coordinates"].str[0]
df["lat"] = df["geometry.coordinates"].str[1]
print(df[["properties.label", "lat", "lon"]])

                 properties.label        lat       lon
0  20 Avenue de Ségur 75007 Paris  48.850699  2.308628


In [ ]:
def geocoder_fr(adresse):
    """Renvoie (latitude, longitude) pour une adresse française, ou (None, None)."""
    url = "https://api-adresse.data.gouv.fr/search/"
    try:
        r = requests.get(url, params={"q": adresse, "limit": 1}, timeout=10)
        r.raise_for_status()
        features = r.json()["features"]
        if not features:
            return None, None
        lon, lat = features[0]["geometry"]["coordinates"]
        return lat, lon
    except requests.exceptions.RequestException:
        return None, None

In [ ]:
print(geocoder_fr("20 avenue de Ségur 75007 Paris"))
print(geocoder_fr("zzzzz qwerty 99999"))

(48.850699, 2.308628)
(None, None)


# 5. Nominatim : le monde entier, une requête par seconde

In [ ]:
url = "https://nominatim.openstreetmap.org/search"
headers = {"User-Agent": "bootcamp-simplon-da/1.0 (prenom.nom@exemple.fr)"}
params = {"q": "54 Via Pietro Mascagni, Catania, Italy", "format": "json", "limit": 1}

r = requests.get(url, params=params, headers=headers, timeout=10)
resultats = r.json()

print(resultats[0]["lat"], resultats[0]["lon"])
print(resultats[0]["display_name"])

37.5111425 15.0950365
Via Pietro Mascagni, Picanello-Ognina-Barriera-Canalicchio, Catania, Sicilia, 95131, Italia


In [ ]:
import pandas as pd
df = pd.json_normalize(resultats)
print(df[["lat", "lon", "display_name"]])

          lat         lon                                       display_name
0  37.5111425  15.0950365  Via Pietro Mascagni, Picanello-Ognina-Barriera...


In [ ]:
import time

def geocoder_monde(adresse):
    """Renvoie (latitude, longitude) n'importe où dans le monde."""
    url = "https://nominatim.openstreetmap.org/search"
    headers = {"User-Agent": "bootcamp-simplon-da/1.0 (prenom.nom@exemple.fr)"}
    params = {"q": adresse, "format": "json", "limit": 1}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        r.raise_for_status()
        resultats = r.json()
        if not resultats:
            return None, None
        return float(resultats[0]["lat"]), float(resultats[0]["lon"])
    except requests.exceptions.RequestException:
        return None, None
    finally:
        time.sleep(1)

In [ ]:
import time
debut = time.time()
print(geocoder_monde("54 Via Pietro Mascagni, Catania, Italy"))
print(geocoder_monde("Piazza del Duomo, Catania, Italy"))
print(f"durée totale : {time.time() - debut:.1f} s")

(37.5111425, 15.0950365)
(37.5024076, 15.0871898)
durée totale : 3.6 s


# 6. Afficher les points sur une carte

In [ ]:
pip install folium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import folium

point = [37.5112, 15.0949]          # latitude, longitude — Catane
m = folium.Map(location=point, zoom_start=14)
folium.Marker(location=point, popup="Un bon restaurant").add_to(m)
m

# Challenge 3 — les restaurants de Catane, sur une carte

In [ ]:
with open("../data/restaurants.json", encoding="utf-8") as f:
    data = json.load(f)

df_restaurants = pd.json_normalize(
    data,
    record_path="restaurants",
    meta=["ville", "pays"]
)

df_restaurants

,nom,adresse,type,notes,ville,pays
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",glacier,"[4.6, 4.8, 4.7]",Catane,Italie
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",trattoria,"[4.4, 4.5]",Catane,Italie
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",poisson,"[4.2, 4.0, 4.3, 4.1]",Catane,Italie
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",grill,"[3.9, 4.1]",Catane,Italie


In [ ]:
moyennes = []

for notes in df_restaurants["notes"]:
    moyenne = sum(notes) / len(notes)
    moyennes.append(round(moyenne, 2))

df_restaurants["note_moyenne"] = moyennes

df_restaurants

,nom,adresse,type,notes,ville,pays,note_moyenne
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",glacier,"[4.6, 4.8, 4.7]",Catane,Italie,4.70
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",trattoria,"[4.4, 4.5]",Catane,Italie,4.45
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",poisson,"[4.2, 4.0, 4.3, 4.1]",Catane,Italie,4.15
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",grill,"[3.9, 4.1]",Catane,Italie,4.00


In [ ]:
def geocoder_monde_honnete(adresse):
    """Renvoie (latitude, longitude) n'importe où dans le monde."""
    url = "https://nominatim.openstreetmap.org/search"
    headers = {"User-Agent": "james_bond@aol.com)"}
    params = {"q": adresse, "format": "json", "limit": 1}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=10)
        r.raise_for_status()
        resultats = r.json()
        if not resultats:
            return None, None
        return float(resultats[0]["lat"]), float(resultats[0]["lon"])
    except requests.exceptions.RequestException:
        return None, None
    finally:
        time.sleep(1)

In [ ]:
latitudes = []
longitudes = []

for adresse in df_restaurants["adresse"]:
    lat, lon = geocoder_monde_honnete(adresse + ", Catania, Italy")
    latitudes.append(lat)
    longitudes.append(lon)

df_restaurants["lat"] = latitudes
df_restaurants["lon"] = longitudes

df_restaurants

,nom,adresse,type,notes,ville,pays,note_moyenne,lat,lon
0,Gelateria Zio Pietro dal 1964,"Via Porta di Ferro, 47, 95131 Catania CT",glacier,"[4.6, 4.8, 4.7]",Catane,Italie,4.70,37.502093,15.093195
1,A Casa d'Amici,"Via Fischetti, 14, 95131 Catania CT",trattoria,"[4.4, 4.5]",Catane,Italie,4.45,37.506171,15.094322
2,La Bitta,"Via Acquicella Porto, 95121 Catania CT",poisson,"[4.2, 4.0, 4.3, 4.1]",Catane,Italie,4.15,37.489440,15.079431
3,Steak House,"Via Porta di Ferro, 8, 95100 Catania CT",grill,"[3.9, 4.1]",Catane,Italie,4.00,37.502933,15.093041


In [ ]:
import folium

point = [df_restaurants["lat"][0], df_restaurants["lon"][0]]

m = folium.Map(location=point, zoom_start=15)

for i in df_restaurants.index:
    nom = df_restaurants["nom"][i]
    note = df_restaurants["note_moyenne"][i]
    lat = df_restaurants["lat"][i]
    lon = df_restaurants["lon"][i]

    folium.Marker(
        location=[lat, lon],
        popup=f"{nom} — {note:.2f}/5"
    ).add_to(m)

m

In [ ]:
df_restaurants.to_csv("../data/restaurants_geocodes.csv", index=False)